# Transcripción y diarización de audio

Pipeline de [`scripts/test_audio_ovh.py`](../scripts/test_audio_ovh.py) paso a paso:

**vídeo → audio → Whisper en OVHcloud (con diarización) → turnos de habla → CSV del contrato → tiempo de habla por hablante**

## 0. Configuración

In [7]:
import os, json, subprocess
from pathlib import Path
import requests
import pandas as pd

VIDEO = "clip_quanta.mp4"   # <- vídeo a analizar
LANG = "ca"                                                       # idioma del audio: ca, es, en...

BASE = os.environ.get("OVH_BASE_URL", "https://oai.endpoints.kepler.ai.cloud.ovh.net/v1")
TOKEN = "".join((os.environ.get("OVH_AI_ENDPOINTS_ACCESS_TOKEN") or os.environ.get("OVH_API_TOKEN") or "").split())
ASR_MODEL = os.environ.get("OVH_ASR_MODEL", "whisper-large-v3")
assert TOKEN, "Falta el token: define la variable de entorno OVH_AI_ENDPOINTS_ACCESS_TOKEN"

## 1. Extraer el audio del vídeo

WAV mono a 16 kHz: es el formato con el que trabaja Whisper y pesa menos al subirlo.

In [8]:
AUDIO = Path(VIDEO).with_suffix(".wav").name
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", VIDEO, "-vn", "-ac", "1", "-ar", "16000", AUDIO], check=True)
print(f"{AUDIO}: {Path(AUDIO).stat().st_size / 1e6:.1f} MB")

clip_quanta.wav: 3.1 MB


## 2. Transcribir con Whisper en OVHcloud

`diarize=true` pide que, además del texto, se indique qué hablante dice cada fragmento. Guardamos la respuesta cruda en JSON para poder revisarla.

In [9]:
with open(AUDIO, "rb") as f:
    r = requests.post(f"{BASE}/audio/transcriptions",
                      headers={"Authorization": f"Bearer {TOKEN}"},
                      files={"file": (AUDIO, f)},
                      data={"model": ASR_MODEL, "language": LANG, "response_format": "verbose_json", "diarize": "true"},
                      timeout=180)
assert r.ok, f"Error {r.status_code}: {r.text[:500]}"
res = r.json()

Path(AUDIO).with_suffix(".json").write_text(json.dumps(res, ensure_ascii=False, indent=2), encoding="utf-8")
list(res.keys())

['task',
 'success',
 'language',
 'duration',
 'text',
 'words',
 'segments',
 'diarization',
 'usage']

## 3. Turnos de habla

OVHcloud devuelve los turnos con su hablante en la clave `diarization` (si no está, usamos `segments`, sin hablante).

In [10]:
turnos = res.get("diarization") or res.get("segments", [])

df = pd.DataFrame([{
    "start": float(s.get("start", 0.0)),
    "end": float(s.get("end", 0.0)),
    "speaker": f"Hablante_{s['speaker']}" if isinstance(s.get("speaker"), int) else str(s.get("speaker", "Desconocido")),
    "text": s.get("text", "").strip(),
} for s in turnos])
print(len(df), "turnos")
df.head(10)

4 turnos


,start,end,speaker,text
0,0.00,12.60,Hablante_0,declarada sobre aquesta qüestió. Ella és la Ca...
1,15.10,51.16,Hablante_1,Carina Givert és catedràtica a la Universitat ...
2,52.94,58.14,Hablante_0,"Carina, què ens pots explicar d'aquestes top s..."
3,58.14,95.96,Hablante_2,"'encanta, m'encanta aquesta introducció que ha..."


## 4. Género de la voz

Clasificador [`prithivMLmods/Common-Voice-Gender-Detection`](https://huggingface.co/prithivMLmods/Common-Voice-Gender-Detection) (código adaptado de su documentación). Recortamos el audio de cada turno con sus tiempos de inicio y fin, lo clasificamos y decidimos una etiqueta por hablante con la media de sus turnos.

In [ ]:
import torch
from scipy.io import wavfile
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor

# Load model and processor
model_name = "prithivMLmods/Common-Voice-Gender-Detection"
model = Wav2Vec2ForSequenceClassification.from_pretrained(model_name)
processor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)

# Label mapping
id2label = {"0": "female", "1": "male"}

# Audio completo del paso 1: ya es mono a 16 kHz, lo que espera el modelo
SR = 16000
_, audio = wavfile.read(AUDIO)
audio = audio.astype("float32") / 32768          # de enteros de 16 bits a decimales entre -1 y 1

def classify_audio(speech):
    """Como en la documentación, pero recibe un trozo de audio (array a 16 kHz) en vez de la ruta a un archivo."""
    inputs = processor(speech, sampling_rate=SR, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.nn.functional.softmax(logits, dim=1).squeeze().tolist()
    return {id2label[str(i)]: round(probs[i], 3) for i in range(len(probs))}

In [ ]:
MIN_DUR = 1.0         # turnos de menos de 1 s: muy poca voz para decidir, los saltamos
MAX_DUR = 15.0        # de los turnos largos usamos solo los primeros 15 s (basta y va más rápido)

def p_female(turno):
    """Probabilidad de voz femenina de un turno, recortando el audio por sus tiempos de inicio y fin."""
    if turno.end - turno.start < MIN_DUR:
        return None
    trozo = audio[int(turno.start * SR): int(min(turno.end, turno.start + MAX_DUR) * SR)]
    return classify_audio(trozo)["female"]

df["p_female"] = df.apply(p_female, axis=1)

In [ ]:
MIN_PROB = 0.80       # por debajo de esta probabilidad media, "no_determinado" (como en el script)

# Una etiqueta por hablante: media de sus turnos (más estable que decidir turno a turno)
p_hablante = df.groupby("speaker").p_female.mean()
genero = p_hablante.map(lambda p: "F" if p >= MIN_PROB else "M" if p <= 1 - MIN_PROB else "no_determinado")
df["genero_inferido"] = df.speaker.map(genero).fillna("no_determinado")
df[["start", "end", "speaker", "p_female", "genero_inferido"]].round(2)

## 5. Exportar el CSV del contrato de datos

Columnas acordadas con el equipo: `tiempo_i`, `tiempo_f`, `transcripcion`, `genero_inferido`.

In [13]:
contrato = (df.rename(columns={"start": "tiempo_i", "end": "tiempo_f", "text": "transcripcion"})
              [["tiempo_i", "tiempo_f", "transcripcion", "genero_inferido"]].round(2))
contrato.to_csv(Path(AUDIO).with_suffix(".csv").name, index=False, encoding="utf-8")
contrato.head()

,tiempo_i,tiempo_f,transcripcion,genero_inferido
0,0.00,12.60,declarada sobre aquesta qüestió. Ella és la Ca...,no_determinado
1,15.10,51.16,Carina Givert és catedràtica a la Universitat ...,no_determinado
2,52.94,58.14,"Carina, què ens pots explicar d'aquestes top s...",no_determinado
3,58.14,95.96,"'encanta, m'encanta aquesta introducció que ha...",no_determinado


## 6. Tiempo de habla por hablante

In [14]:
dur = (df.end - df.start).clip(lower=0)
tiempos = dur.groupby(df.speaker).sum().sort_values(ascending=False)

resumen = pd.DataFrame({"segundos": tiempos.round(1), "%": (100 * tiempos / tiempos.sum()).round(1)})
print(f"Total voz detectada: {tiempos.sum():.1f} s")
resumen

Total voz detectada: 91.7 s


,segundos,%
speaker,,
Hablante_2,37.8,41.3
Hablante_1,36.1,39.3
Hablante_0,17.8,19.4
